# GMM and UMAP+GMM Exploration
**Features:** Set B — band colour differences + log-transformed redshift  
**Classes:** STAR, GALAXY, QSO  

Labels are used **only for post-hoc validation** (ARI, NMI, F1). They play no role in model selection or cluster assignment.

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (
    f1_score, adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    confusion_matrix
)

import umap
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os

sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('Set_D.csv')

FEATURES = ['u-g', 'g-r', 'r-i', 'i-z', 'redshift_log']
X = df[FEATURES].values
y = df['class'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)   # held back — only used at evaluation
class_names = le.classes_

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'X shape: {X_scaled.shape}')
print(f'Classes: {class_names}')
print(pd.Series(y).value_counts().to_string())

In [ ]:
# Post-hoc evaluation helper
# Uses linear_sum_assignment to find the optimal permutation of cluster IDs -> class IDs
# that maximises accuracy. Labels are only touched here, after clustering is complete.
def evaluate_clusters(y_true, cluster_labels, X, n_classes=3):
    """Returns metrics and optimally-mapped predicted labels."""
    n_clusters = len(set(cluster_labels))

    # Build cost matrix: rows = true classes, cols = cluster IDs
    cost = np.zeros((n_classes, n_clusters), dtype=int)
    for true_cls in range(n_classes):
        for clust in range(n_clusters):
            cost[true_cls, clust] = np.sum((y_true == true_cls) & (cluster_labels == clust))

    # Optimal assignment (maximise overlap -> negate for minimisation)
    row_ind, col_ind = linear_sum_assignment(-cost)
    mapping = {col: row for row, col in zip(row_ind, col_ind)}

    # Map cluster IDs to class IDs
    y_pred = np.array([mapping.get(c, -1) for c in cluster_labels])

    ari = adjusted_rand_score(y_true, cluster_labels)
    nmi = normalized_mutual_info_score(y_true, cluster_labels)
    f1  = f1_score(y_true, y_pred, average='macro')
    sil = silhouette_score(X, cluster_labels)
    ch  = calinski_harabasz_score(X, cluster_labels)
    db  = davies_bouldin_score(X, cluster_labels)

    return {'f1': f1, 'ari': ari, 'nmi': nmi, 'sil': sil, 'ch': ch, 'db': db,
            'y_pred': y_pred, 'mapping': mapping}

print('evaluate_clusters helper ready.')

---
## Part 1 — Gaussian Mixture Model (GMM)

GMM fits a mixture of Gaussians to the data. Each component has its own mean and covariance, allowing elliptical, non-spherical cluster shapes — well suited to the overlapping distributions in colour-redshift space.

**Model selection uses BIC only** — a purely unsupervised criterion that rewards fit while penalising complexity. Labels are not seen until the final evaluation block.

In [ ]:
n_components_range = range(3, 16)
cov_types = ['full', 'tied', 'diag', 'spherical']

results = []
best_bic  = np.inf
best_gmm  = None
best_gmm_labels = None

for cov in cov_types:
    for n in n_components_range:
        gmm = GaussianMixture(n_components=n, covariance_type=cov,
                              random_state=42, max_iter=300, n_init=3)
        gmm.fit(X_scaled)                      # no labels used
        bic = gmm.bic(X_scaled)
        aic = gmm.aic(X_scaled)
        labels = gmm.predict(X_scaled)

        # Internal metrics only
        sil = silhouette_score(X_scaled, labels)
        ch  = calinski_harabasz_score(X_scaled, labels)
        db  = davies_bouldin_score(X_scaled, labels)
        results.append({'n': n, 'cov': cov, 'bic': bic, 'aic': aic,
                        'silhouette': sil, 'CH': ch, 'DB': db})

        if bic < best_bic:
            best_bic = bic
            best_gmm = gmm
            best_gmm_labels = labels.copy()

results_df = pd.DataFrame(results)
print(f'Best by BIC: n={best_gmm.n_components}, cov={best_gmm.covariance_type}, BIC={best_bic:.1f}')

In [ ]:
# BIC / AIC curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for cov in cov_types:
    sub = results_df[results_df['cov'] == cov]
    axes[0].plot(sub['n'], sub['bic'], marker='o', markersize=4, label=cov)
    axes[1].plot(sub['n'], sub['aic'], marker='o', markersize=4, label=cov)
for ax, title in zip(axes, ['BIC', 'AIC']):
    ax.set_xlabel('n_components', fontsize=10)
    ax.set_ylabel(title, fontsize=10)
    ax.set_title(f'GMM {title} vs n_components', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
plt.suptitle('GMM Model Selection (BIC / AIC) — no labels used', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Post-hoc validation — labels used here for the first time
gmm_eval = evaluate_clusters(y_enc, best_gmm_labels, X_scaled)

print(f'Best GMM: n_components={best_gmm.n_components}, covariance_type={best_gmm.covariance_type}')
print(f'\n--- Internal metrics (used for selection) ---')
print(f'Silhouette:          {gmm_eval["sil"]:.4f}')
print(f'Calinski-Harabasz:   {gmm_eval["ch"]:.4f}')
print(f'Davies-Bouldin:      {gmm_eval["db"]:.4f}')
print(f'\n--- External validation (labels used here only) ---')
print(f'F1 (macro):          {gmm_eval["f1"]:.4f}')
print(f'ARI:                 {gmm_eval["ari"]:.4f}')
print(f'NMI:                 {gmm_eval["nmi"]:.4f}')

In [ ]:
# Soft assignment heatmap + PCA visualisation
probs = best_gmm.predict_proba(X_scaled)
sample_idx = np.random.default_rng(42).choice(len(X_scaled), size=500, replace=False)
sample_idx = sample_idx[np.argsort(y_enc[sample_idx])]

plt.figure(figsize=(12, 4))
sns.heatmap(probs[sample_idx], cmap='Blues', vmin=0, vmax=1, yticklabels=False,
            xticklabels=[f'C{i}' for i in range(best_gmm.n_components)],
            cbar_kws={'label': 'Probability'})
plt.xlabel('GMM Component', fontsize=10)
plt.ylabel('Sample (sorted by true class, n=500)', fontsize=10)
plt.title(f'Soft Assignments — GMM (n={best_gmm.n_components}, {best_gmm.covariance_type})', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# PCA 2D: true labels vs GMM components
X2d = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
cmap_cls  = plt.cm.get_cmap('tab10', len(class_names))
cmap_comp = plt.cm.get_cmap('tab20', best_gmm.n_components)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, cls in enumerate(class_names):
    mask = y_enc == i
    axes[0].scatter(X2d[mask,0], X2d[mask,1], c=[cmap_cls(i)], s=5, alpha=0.5, label=cls, linewidths=0)
axes[0].set_title('True Labels (PCA 2D)', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9, markerscale=3)
axes[0].grid(True, alpha=0.3)

for k in range(best_gmm.n_components):
    mask = best_gmm_labels == k
    axes[1].scatter(X2d[mask,0], X2d[mask,1], c=[cmap_comp(k)], s=5, alpha=0.5, linewidths=0)
axes[1].set_title(f'GMM Components (n={best_gmm.n_components}, {best_gmm.covariance_type})', fontsize=11, fontweight='bold')
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlabel('PC1', fontsize=9); ax.set_ylabel('PC2', fontsize=9)

plt.suptitle(f'GMM | F1={gmm_eval["f1"]:.4f} | ARI={gmm_eval["ari"]:.4f} | NMI={gmm_eval["nmi"]:.4f}',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

os.makedirs('models', exist_ok=True)
joblib.dump(best_gmm, 'models/gmm_model.pkl')
print('Saved models/gmm_model.pkl')

---
## Part 2 — UMAP + GMM

UMAP (Uniform Manifold Approximation and Projection) learns a 2D manifold that preserves local neighbourhood structure from the original 5D feature space. GMM then clusters the compressed embedding.

The motivation: UMAP may untangle class structure that overlaps in the original space (particularly the STAR/GALAXY boundary), giving GMM a better-separated input. Both steps are fully unsupervised — labels are withheld until the final evaluation block.

Note: since kNN requires labels for training, it was replaced here with GMM on the UMAP embedding to keep the pipeline label-free throughout.

In [ ]:
# Grid search: UMAP params x GMM covariance type
# Selection criterion: BIC of GMM fit on the UMAP embedding (no labels)
umap_n_neighbors = [5, 15, 30, 50]
umap_min_dist    = [0.0, 0.1, 0.3]
gmm_cov_types    = ['full', 'tied', 'diag']

best_bic_umap  = np.inf
best_cfg_umap  = None
best_embedding = None
best_umap_gmm  = None
best_umap_labels = None

for nn in umap_n_neighbors:
    for md in umap_min_dist:
        reducer = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=md, random_state=42)
        embedding = reducer.fit_transform(X_scaled)   # no labels

        for cov in gmm_cov_types:
            gmm = GaussianMixture(n_components=3, covariance_type=cov,
                                  random_state=42, max_iter=300, n_init=3)
            gmm.fit(embedding)
            bic = gmm.bic(embedding)

            if bic < best_bic_umap:
                best_bic_umap  = bic
                best_cfg_umap  = {'n_neighbors': nn, 'min_dist': md, 'cov': cov}
                best_embedding = embedding.copy()
                best_umap_gmm  = gmm
                best_umap_labels = gmm.predict(embedding).copy()

print(f'Best UMAP+GMM by BIC: {best_cfg_umap}, BIC={best_bic_umap:.1f}')

In [ ]:
# Post-hoc validation
umap_eval = evaluate_clusters(y_enc, best_umap_labels, best_embedding)

print(f'UMAP config: n_neighbors={best_cfg_umap["n_neighbors"]}, min_dist={best_cfg_umap["min_dist"]}')
print(f'GMM config:  n_components=3, covariance_type={best_cfg_umap["cov"]}')
print(f'\n--- Internal metrics ---')
print(f'Silhouette:          {umap_eval["sil"]:.4f}')
print(f'Calinski-Harabasz:   {umap_eval["ch"]:.4f}')
print(f'Davies-Bouldin:      {umap_eval["db"]:.4f}')
print(f'\n--- External validation ---')
print(f'F1 (macro):          {umap_eval["f1"]:.4f}')
print(f'ARI:                 {umap_eval["ari"]:.4f}')
print(f'NMI:                 {umap_eval["nmi"]:.4f}')

In [ ]:
# Confusion matrix (post-hoc)
cm = confusion_matrix(y_enc, umap_eval['y_pred'])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={'fontweight': 'bold'})
plt.xlabel('Predicted (optimal assignment)', fontsize=10)
plt.ylabel('True', fontsize=10)
plt.title('UMAP+GMM — Post-hoc Confusion Matrix', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# UMAP scatter: true labels vs GMM clusters
cmap_cls  = plt.cm.get_cmap('tab10', len(class_names))
cmap_clust = plt.cm.get_cmap('tab10', 3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, cls in enumerate(class_names):
    mask = y_enc == i
    axes[0].scatter(best_embedding[mask,0], best_embedding[mask,1],
                    c=[cmap_cls(i)], s=5, alpha=0.5, label=cls, linewidths=0)
axes[0].set_title('True Labels (UMAP 2D)', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9, markerscale=3)
axes[0].grid(True, alpha=0.3)

for k in range(3):
    mask = best_umap_labels == k
    axes[1].scatter(best_embedding[mask,0], best_embedding[mask,1],
                    c=[cmap_clust(k)], s=5, alpha=0.5,
                    label=f'Cluster {k}', linewidths=0)
axes[1].set_title(f'GMM Clusters on UMAP Embedding', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9, markerscale=3)
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlabel('UMAP 1', fontsize=9); ax.set_ylabel('UMAP 2', fontsize=9)

plt.suptitle(
    f'UMAP+GMM | F1={umap_eval["f1"]:.4f} | ARI={umap_eval["ari"]:.4f} | NMI={umap_eval["nmi"]:.4f}',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()

joblib.dump(best_umap_gmm, 'models/umap_gmm_model.pkl')
print('Saved models/umap_gmm_model.pkl')